In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 2 — Install (restart kernel after this cell)
import subprocess
subprocess.run(['pip', 'install', '-q',
    'transformers==4.46.3',
    'datasets',
    'seqeval',
    'accelerate',
])
print('Done. Restart kernel now, then run all cells below.')

Done. Restart kernel now, then run all cells below.


In [ ]:
# Cell 3 — Imports
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset, Dataset as HFDataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import f1_score, classification_report

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print('Imports OK')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

Imports OK
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
Free: 15.5 GB


In [ ]:
# Cell 4 — Label maps
LABEL_LIST = ['O', 'B-SKILL', 'I-SKILL']
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for i, l in enumerate(LABEL_LIST)}
print('Labels:', LABEL2ID)

Labels: {'O': 0, 'B-SKILL': 1, 'I-SKILL': 2}


In [ ]:
# Cell 5 — Tag normalisation (merges tags_skill + tags_knowledge)
def normalise_tag(tag):
    if tag in ('B', 'B-SOFT', 'B-ICT', 'B-SKILL',
               'B-PENSEE', 'B-RESULTATS', 'B-RELATIONNEL', 'B-PERSONNEL'):
        return 'B-SKILL'
    if tag in ('I', 'I-SOFT', 'I-ICT', 'I-SKILL',
               'I-PENSEE', 'I-RESULTATS', 'I-RELATIONNEL', 'I-PERSONNEL'):
        return 'I-SKILL'
    return 'O'

def get_tokens_and_tags(example):
    """Merges tags_skill + tags_knowledge (where present) into B-SKILL/I-SKILL/O."""
    tokens = example['tokens']
    merged = ['O'] * len(tokens)
    for col in ('tags_skill', 'tags_knowledge'):
        if col not in example:
            continue
        for i, tag in enumerate(example[col]):
            norm = normalise_tag(str(tag))
            if norm != 'O' and merged[i] == 'O':
                merged[i] = norm
    return tokens, merged

print('Tag normalisation defined')

Tag normalisation defined


In [ ]:
# Cell 6 — Load all 4 English datasets + combine into single train/test split
say = load_dataset('jjzha/sayfullina')
skl = load_dataset('jjzha/skillspan')
grn = load_dataset('jjzha/green')
gnh = load_dataset('jjzha/gnehm')

ALL_DATASETS = [
    ('sayfullina', say),
    ('skillspan',  skl),
    ('green',      grn),
    ('gnehm',      gnh),
]

print('Individual dataset sizes:')
for name, ds in ALL_DATASETS:
    skill_count = sum(
        1 for ex in ds['train']
        if any(t != 'O' for t in get_tokens_and_tags(ex)[1])
    )
    print(f'  {name:12} train={len(ds["train"]):5}  test={len(ds["test"]):4}  with_skills={skill_count}')

print(f'\nTotal train: {sum(len(ds["train"]) for _, ds in ALL_DATASETS)}')
print(f'Total test:  {sum(len(ds["test"])  for _, ds in ALL_DATASETS)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

...

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/19889 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2557 [00:00<?, ? examples/s]

Individual dataset sizes:
  sayfullina   train= 3705  test=1851  with_skills=3701
  skillspan    train= 4800  test=3569  with_skills=1584
  green        train= 8669  test= 335  with_skills=5136
  gnehm        train=19889  test=2557  with_skills=3408

Total train: 37063
Total test:  8312


In [ ]:
# Cell 7 — Tokenise + align labels
def tokenize_and_align(examples, tokenizer, max_length=128):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=max_length,
        is_split_into_words=True,
    )
    all_labels = []
    for i in range(len(examples['tokens'])):
        word_ids  = tokenized.word_ids(batch_index=i)
        tags      = examples['ner_tags'][i]
        label_ids = []
        prev_word = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(LABEL2ID[tags[word_id]])
            else:
                label_ids.append(-100)
            prev_word = word_id
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

def build_hf_split(ds_split, tokenizer, desc=''):
    records = []
    for ex in tqdm(ds_split, desc=desc):
        tokens, tags = get_tokens_and_tags(ex)
        records.append({'tokens': tokens, 'ner_tags': tags})
    hf = HFDataset.from_list(records)
    hf = hf.map(
        lambda batch: tokenize_and_align(batch, tokenizer),
        batched=True,
        remove_columns=['tokens', 'ner_tags'],
    )
    return hf

print('Tokenisation defined')

Tokenisation defined


In [ ]:
# Cell 8 — Compute metrics (seqeval strict F1)
def make_compute_metrics():
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        true_labels = [
            [ID2LABEL[l] for l in label_row if l != -100]
            for label_row in labels
        ]
        true_preds = [
            [ID2LABEL[p] for p, l in zip(pred_row, label_row) if l != -100]
            for pred_row, label_row in zip(predictions, labels)
        ]
        return {'f1': f1_score(true_labels, true_preds)}
    return compute_metrics

print('Metrics defined')

Metrics defined


In [ ]:
# Cell 9 — Models to train (one combined model per architecture)
MODELS = {
    'bert-base':    'bert-base-cased',
    'spanbert':     'SpanBERT/spanbert-base-cased',
    'jobbert':      'jjzha/jobbert-base-cased',
    'jobspanbert':  'jjzha/jobspanbert-base-cased',
}
print('Models to train:')
for k, v in MODELS.items():
    print(f'  {k:14} → {v}')

Models to train:
  bert-base      → bert-base-cased
  spanbert       → SpanBERT/spanbert-base-cased
  jobbert        → jjzha/jobbert-base-cased
  jobspanbert    → jjzha/jobspanbert-base-cased


In [ ]:
# Cell 10 — Training loop: one combined model per architecture
# Trains on all 4 datasets concatenated → single model per architecture
# Saves to Google Drive

RESULTS = {}  # model_name → {overall_f1, per_dataset_f1}

for model_name, model_id in MODELS.items():
    print(f'\n{"="*60}')
    print(f'MODEL: {model_name} ({model_id})')
    print(f'{"="*60}')

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Build combined train split from all 4 datasets
    print('Building combined training set...')
    all_train_records = []
    for name, ds in ALL_DATASETS:
        for ex in tqdm(ds['train'], desc=f'  {name}'):
            tokens, tags = get_tokens_and_tags(ex)
            all_train_records.append({'tokens': tokens, 'ner_tags': tags})

    # Shuffle combined training set
    random.shuffle(all_train_records)
    combined_train_hf = HFDataset.from_list(all_train_records)
    combined_train_hf = combined_train_hf.map(
        lambda batch: tokenize_and_align(batch, tokenizer),
        batched=True,
        remove_columns=['tokens', 'ner_tags'],
    )
    print(f'Combined train size: {len(combined_train_hf)}')

    # Build combined test split
    print('Building combined test set...')
    all_test_records = []
    for name, ds in ALL_DATASETS:
        for ex in tqdm(ds['test'], desc=f'  {name}'):
            tokens, tags = get_tokens_and_tags(ex)
            all_test_records.append({'tokens': tokens, 'ner_tags': tags})

    combined_test_hf = HFDataset.from_list(all_test_records)
    combined_test_hf = combined_test_hf.map(
        lambda batch: tokenize_and_align(batch, tokenizer),
        batched=True,
        remove_columns=['tokens', 'ner_tags'],
    )
    print(f'Combined test size:  {len(combined_test_hf)}')

    model = AutoModelForTokenClassification.from_pretrained(
        model_id,
        num_labels=len(LABEL_LIST),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    save_dir = f'/content/drive/MyDrive/phaseA_combined/{model_name}'
    os.makedirs(save_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir                  = save_dir,
        num_train_epochs            = 3,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 32,
        learning_rate               = 2e-5,
        weight_decay                = 0.01,
        warmup_ratio                = 0.1,
        evaluation_strategy         = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_f1',
        fp16                        = True,
        logging_steps               = 50,
        report_to                   = 'none',
        seed                        = 42,
    )

    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = combined_train_hf,
        eval_dataset    = combined_test_hf,
        tokenizer       = tokenizer,
        data_collator   = DataCollatorForTokenClassification(tokenizer),
        compute_metrics = make_compute_metrics(),
    )

    trainer.train()

    metrics = trainer.evaluate()
    overall_f1 = metrics['eval_f1']
    print(f'\n{model_name} combined F1: {overall_f1:.4f}')

    trainer.save_model(save_dir + '/best')
    tokenizer.save_pretrained(save_dir + '/best')
    print(f'Saved to {save_dir}/best')

    RESULTS[model_name] = overall_f1

    del model, trainer, combined_train_hf, combined_test_hf
    gc.collect()
    torch.cuda.empty_cache()

print('\nAll models trained.')


MODEL: bert-base (bert-base-cased)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Combined test size:  8312


model.safetensors:   0%|          | 0.00/539M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at jjzha/jobspanbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_1528/299690349.py:77: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1
1,0.253300,0.148631,0.570110
2,0.175400,0.130001,0.633523
3,0.142000,0.129585,0.653658



jobspanbert combined F1: 0.6537
Saved to /content/drive/MyDrive/phaseA_combined/jobspanbert/best

All models trained.


In [ ]:
# Cell 11 — Per-dataset F1 breakdown for each saved model
# Loads each saved model and evaluates on each dataset's test split separately

per_dataset_results = {}  # (model_name, ds_name) → f1

for model_name in MODELS:
    model_path = f'/content/drive/MyDrive/phaseA_combined/{model_name}/best'
    print(f'\nEvaluating {model_name}...')

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    model.eval()

    for ds_name, ds in ALL_DATASETS:
        test_hf = build_hf_split(ds['test'], tokenizer, desc=f'  {ds_name}')

        eval_args = TrainingArguments(
            output_dir='/tmp/eval',
            per_device_eval_batch_size=32,
            report_to='none',
            fp16=torch.cuda.is_available(),
        )
        evaluator = Trainer(
            model           = model,
            args            = eval_args,
            eval_dataset    = test_hf,
            tokenizer       = tokenizer,
            data_collator   = DataCollatorForTokenClassification(tokenizer),
            compute_metrics = make_compute_metrics(),
        )
        metrics = evaluator.evaluate()
        f1 = metrics['eval_f1']
        per_dataset_results[(model_name, ds_name)] = f1
        print(f'  {ds_name:12} F1 = {f1:.4f}')

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


Evaluating bert-base...


  sayfullina: 100%|██████████| 1851/1851 [00:00<00:00, 8026.23it/s]


Map:   0%|          | 0/1851 [00:00<?, ? examples/s]

/tmp/ipykernel_1528/4129834275.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  evaluator = Trainer(


  sayfullina   F1 = 0.8278


...

  green        F1 = 0.4085


  gnehm: 100%|██████████| 2557/2557 [00:00<00:00, 14577.91it/s]


Map:   0%|          | 0/2557 [00:00<?, ? examples/s]

/tmp/ipykernel_1528/4129834275.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  evaluator = Trainer(


  gnehm        F1 = 0.7828


In [ ]:
# Cell 12 — Results summary table + save to Drive
ds_names    = [n for n, _ in ALL_DATASETS]
model_names = list(MODELS.keys())

print(f'{"Model":<15}', end='')
for ds in ds_names:
    print(f'{ds:>12}', end='')
print(f'{"Avg":>8}')
print('-' * (15 + 12 * len(ds_names) + 8))

for model_name in model_names:
    f1s = [per_dataset_results.get((model_name, ds), 0.0) for ds in ds_names]
    avg = sum(f1s) / len(f1s)
    print(f'{model_name:<15}', end='')
    for f1 in f1s:
        print(f'{f1:>12.4f}', end='')
    print(f'{avg:>8.4f}')

# Save to Drive
rows = []
for (model_name, ds_name), f1 in per_dataset_results.items():
    rows.append({'model': model_name, 'dataset': ds_name, 'f1': f1})
df = pd.DataFrame(rows)
os.makedirs('/content/drive/MyDrive/phaseA_combined', exist_ok=True)
df.to_csv('/content/drive/MyDrive/phaseA_combined/results.csv', index=False)
print('\nSaved results.csv to Drive')

Model            sayfullina   skillspan       green       gnehm     Avg
-----------------------------------------------------------------------
bert-base            0.8278      0.5014      0.4459      0.8260  0.6503
spanbert             0.8003      0.4765      0.3983      0.7913  0.6166
jobbert              0.8091      0.5429      0.4079      0.8070  0.6417
jobspanbert          0.8207      0.5329      0.4085      0.7828  0.6362

Saved results.csv to Drive


In [ ]:
# Cell 12 — Per-dataset precision, recall, F1 + save token-level predictions to Drive

from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
import numpy as np
import pandas as pd
import os

def get_predictions(model, tokenizer, hf_dataset):
    """Run inference on a tokenised HF dataset, return true + pred label sequences."""
    eval_args = TrainingArguments(
        output_dir='/tmp/eval',
        per_device_eval_batch_size=32,
        report_to='none',
        fp16=torch.cuda.is_available(),
    )
    evaluator = Trainer(
        model           = model,
        args            = eval_args,
        eval_dataset    = hf_dataset,
        tokenizer       = tokenizer,
        data_collator   = DataCollatorForTokenClassification(tokenizer),
        compute_metrics = make_compute_metrics(),
    )
    output = evaluator.predict(hf_dataset)
    logits, labels = output.predictions, output.label_ids
    predictions = np.argmax(logits, axis=-1)

    true_seqs = [
        [ID2LABEL[l] for l in label_row if l != -100]
        for label_row in labels
    ]
    pred_seqs = [
        [ID2LABEL[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]
    return true_seqs, pred_seqs


all_rows = []           # token-level rows for CSV
summary_rows = []       # per-model per-dataset P/R/F1

for model_name in MODELS:
    model_path = f'/content/drive/MyDrive/phaseA_combined/{model_name}/best'
    print(f'\n{"="*55}')
    print(f'Model: {model_name}')
    print(f'{"="*55}')

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model     = AutoModelForTokenClassification.from_pretrained(model_path)
    model.eval()

    for ds_name, ds in ALL_DATASETS:
        # Build per-dataset test split (keep raw tokens for CSV)
        raw_records = []
        for ex in tqdm(ds['test'], desc=f'  Preparing {ds_name}'):
            tokens, tags = get_tokens_and_tags(ex)
            raw_records.append({'tokens': tokens, 'ner_tags': tags})

        test_hf = HFDataset.from_list(raw_records)
        test_hf_tok = test_hf.map(
            lambda batch: tokenize_and_align(batch, tokenizer),
            batched=True,
            remove_columns=['tokens', 'ner_tags'],
        )

        true_seqs, pred_seqs = get_predictions(model, tokenizer, test_hf_tok)

        p  = precision_score(true_seqs, pred_seqs)
        r  = recall_score(true_seqs, pred_seqs)
        f1 = f1_score(true_seqs, pred_seqs)

        print(f'\n  {ds_name}')
        print(f'    Precision: {p:.4f}')
        print(f'    Recall:    {r:.4f}')
        print(f'    F1:        {f1:.4f}')
        print(classification_report(true_seqs, pred_seqs))

        summary_rows.append({
            'model':     model_name,
            'dataset':   ds_name,
            'precision': round(p,  4),
            'recall':    round(r,  4),
            'f1':        round(f1, 4),
        })

        # Token-level rows
        example_id = 0
        for raw, true_seq, pred_seq in zip(raw_records, true_seqs, pred_seqs):
            tokens = raw['tokens']
            for token_id, (token, gold, pred) in enumerate(zip(tokens, true_seq, pred_seq)):
                all_rows.append({
                    'model':      model_name,
                    'dataset':    ds_name,
                    'example_id': example_id,
                    'token_id':   token_id,
                    'token':      token,
                    'gold':       gold,
                    'pred':       pred,
                })
            example_id += 1

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# Save summary P/R/F1
summary_df = pd.DataFrame(summary_rows)
print('\n\n=== SUMMARY ===')
print(summary_df.to_string(index=False))
os.makedirs('/content/drive/MyDrive/phaseA_combined', exist_ok=True)
summary_df.to_csv('/content/drive/MyDrive/phaseA_combined/results_prf.csv', index=False)
print('\nSaved results_prf.csv')

# Save token-level predictions
tokens_df = pd.DataFrame(all_rows)
tokens_df.to_csv('/content/drive/MyDrive/phaseA_combined/predictions_tokens.csv', index=False)
print(f'Saved predictions_tokens.csv  ({len(tokens_df)} rows)')
tokens_df.head(20)



Model: bert-base


  Preparing sayfullina: 100%|██████████| 1851/1851 [00:00<00:00, 2684.76it/s]


Map:   0%|          | 0/1851 [00:00<?, ? examples/s]

/tmp/ipykernel_1528/132801876.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  evaluator = Trainer(



  sayfullina
    Precision: 0.8007
    Recall:    0.8568
    F1:        0.8278
              precision    recall  f1-score   support

       SKILL       0.80      0.86      0.83      1857

   micro avg       0.80      0.86      0.83      1857
   macro avg       0.80      0.86      0.83      1857
weighted avg       0.80      0.86      0.83      1857



...

  Preparing gnehm: 100%|██████████| 2557/2557 [00:00<00:00, 14553.67it/s]


Map:   0%|          | 0/2557 [00:00<?, ? examples/s]

/tmp/ipykernel_1528/132801876.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  evaluator = Trainer(



  gnehm
    Precision: 0.7712
    Recall:    0.7948
    F1:        0.7828
              precision    recall  f1-score   support

       SKILL       0.77      0.79      0.78       848

   micro avg       0.77      0.79      0.78       848
   macro avg       0.77      0.79      0.78       848
weighted avg       0.77      0.79      0.78       848



=== SUMMARY ===
      model    dataset  precision  recall     f1
  bert-base sayfullina     0.8007  0.8568 0.8278
  bert-base  skillspan     0.4894  0.5141 0.5014
  bert-base      green     0.4991  0.4030 0.4459
  bert-base      gnehm     0.8164  0.8359 0.8260
   spanbert sayfullina     0.7795  0.8223 0.8003
   spanbert  skillspan     0.4761  0.4770 0.4765
   spanbert      green     0.4566  0.3532 0.3983
   spanbert      gnehm     0.7816  0.8012 0.7913
    jobbert sayfullina     0.7836  0.8363 0.8091
    jobbert  skillspan     0.5194  0.5687 0.5429
    jobbert      green     0.4542  0.3701 0.4079
    jobbert      gnehm     0.7842  0.8311 0.80

,model,dataset,example_id,token_id,token,gold,pred
0,bert-base,sayfullina,0,0,be,O,O
1,bert-base,sayfullina,0,1,look,O,O
2,bert-base,sayfullina,0,2,for,O,O
3,bert-base,sayfullina,0,3,a,O,O
4,bert-base,sayfullina,0,4,temporary,O,O
5,bert-base,sayfullina,0,5,opportunity,O,O
6,bert-base,sayfullina,0,6,within,O,O
7,bert-base,sayfullina,0,7,a,O,O
8,bert-base,sayfullina,0,8,progressive,O,O
9,bert-base,sayfullina,0,9,and,O,O
